<a href="https://colab.research.google.com/github/dxda6216/ttron2excel/blob/main/ttron_data_file_to_excel_file_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###############################################################################
# CELL 1 - Settings (Colab form)
###############################################################################

# @title Converting a Taylortron TRACES file (TRACES.nnn) to Excel files
# @markdown **This script works only with a specific format of data files (*TRACES.nnn* files) generated by the [Taylortron](https://doi.org/10.1080/09291018209359765) in the Carl Johnson Lab.**

# @markdown 1. Input the experiment number (avoid spaces and special characters).
# @markdown 2. Input the experiment title (this field can be blank).
# @markdown 3. Input the date on which the experiment started.
# @markdown 4. Select 'Sinc Filter' ([firwin](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html)) or '[Moving Average](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)' for detrending.
# @markdown 5. If 'Sinc Filter' is chosen, adjust `Sinc_Filter_Cutoff_Period_Hours` and `Sinc_Filter_Order`.
# @markdown 6. If 'Moving Average' is chosen, adjust `Window_size_for_trend_line_moving_average`.
# @markdown 7. **Runtime** -> **Restart and run all** (or press **Ctrl+M** and then **Ctrl+F9**).
# @markdown 8. Wait until the `Choose Files` / `Browse...` button appears below.
# @markdown 9. Click it and select the *TRACES.nnn* file on your computer.
# @markdown 10. Wait a while. Three Excel files, one ZIP file and one PDF file will be saved to your "Downloads" folder.

# @markdown - The first Excel file has multiple sheets containing the raw data, smoothed data, detrended data and the detected peak and trough times.
# @markdown - The second Excel file has a single sheet containing only the raw time series. The sampling interval is given in the sheet name. This file can be opened with programs such as [pyBOAT](https://github.com/tensionhead/pyBOAT).
# @markdown - The third Excel file has a single sheet containing the damped-sine fitting results.
# @markdown - The ZIP file contains one .dat file per channel, readable by the [LumiCycle](https://actimetrics.com/products/lumicycle/) Analysis program.

# @markdown ---
# @markdown **Experiment**
Experiment_number = 'CYxxx'  # @param {type:"string"}
Experiment_title = ''  # @param {type:"string"}
Date_experiment_started = '2026-01-01'  # @param {type:"date"}

# @markdown **Detrending method**
Detrending_Method = "Sinc Filter"  # @param ["Sinc Filter", "Moving Average"]

# @markdown **Detrending parameters**
Sinc_Filter_Cutoff_Period_Hours = 48  # @param {type:"slider", min:1, max:240, step:1}
Sinc_Filter_Order = 101  # @param {type:"slider", min:1, max:361, step:2}
Window_size_for_trend_line_moving_average = 24  # @param {type:"slider", min:1, max:120, step:1}

# @markdown **Overall (multi-panel) plots**
Data_Plotting = "Plotting the channel 00 data last"  # @param ["Plotting the channel 00 data first", "Plotting the channel 00 data last"]
Subplot_Matrix = "3-column by 10-row"  # @param ["3-column by 10-row", "4-column by 8-row"]

# @markdown **X-axis ticks**
Major_Ticks = "Every 24 hours"  # @param ["Every 12 hours", "Every 24 hours", "Every 48 hours"]
Minor_Ticks = "Every 12 hours"  # @param ["No minor ticks", "Every 2 hours", "Every 4 hours", "Every 6 hours", "Every 12 hours", "Every 24 hours"]

# @markdown **Peak / trough labels**
Label_peaks_and_troughs_in_detrended_data_plot = "Yes"  # @param ["Yes", "No"]
Label_peaks_and_troughs_in_actogram = "Yes"  # @param ["Yes", "No"]

# @markdown **Double-plot actogram**
Actogram_X_axis_scale = 24  # @param {type:"slider", min:12, max:60, step:0.1}

# @markdown **Time range (in hours) used for the damped sine fit**
Start_Hour_for_Fitting = 24  # @param {type:"slider", min:0.0, max:360.0, step:1}
End_Hour_for_Fitting = 120  # @param {type:"slider", min:0.0, max:360.0, step:1}


###############################################################################
# CELL 2 - Setup: imports and helper functions
###############################################################################

# @title Setup: imports and helper functions (no user input needed) { display-mode: "form" }
"""Helpers for turning a Taylortron TRACES.nnn file into Excel workbooks,
per-channel .dat files and a multi-page PDF of plots.

Every function here takes what it needs as arguments, so this cell can be run
before the settings above are chosen and does not depend on any global state.
"""

import math
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import FormatStrFormatter, MultipleLocator
from scipy import signal
from scipy.optimize import curve_fit

try:  # running on Colab
    from google.colab import files as colab_files

    IN_COLAB = True
except ImportError:  # running on a local Jupyter kernel
    colab_files = None
    IN_COLAB = False

# --------------------------------------------------------------------------- #
# Constants
# --------------------------------------------------------------------------- #

N_CHANNELS = 30
CHANNELS = [f"{k:02d}" for k in range(N_CHANNELS)]
COLUMN_NAMES = ["Hours"] + CHANNELS

HOURS_PER_DAY = 24.0
PEAK_MIN_SEPARATION_HOURS = 12.0   # minimum spacing between detected peaks
SMOOTHING_WINDOWS = (5, 9)         # moving-average windows reported in the workbook
FIT_MIN_PERIOD_HOURS = 12.0        # bounds for the damped sine period
FIT_MAX_PERIOD_HOURS = 60.0
FIT_MIN_POINTS = 5                 # one point per fitted parameter
EXCEL_SHEET_NAME_LIMIT = 31        # hard limit imposed by the .xlsx format


# --------------------------------------------------------------------------- #
# Small utilities
# --------------------------------------------------------------------------- #

def sheet_name(name: str) -> str:
    """Return a sheet name Excel will accept (<= 31 chars, no []:*?/\\)."""
    for bad in "[]:*?/\\":
        name = name.replace(bad, "-")
    return name[:EXCEL_SHEET_NAME_LIMIT]


def upper_ylimit(series: pd.Series, headroom: float = 1.10) -> float:
    """Top of the y-axis for a channel, safe for empty or all-NaN channels."""
    if not series.notna().any():
        return 1.0
    return max(float(np.nanmax(series.to_numpy())) * headroom, 1.0)


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def utc_now_string() -> str:
    return utc_now().strftime("%Y-%m-%d %H:%M:%S")


# --------------------------------------------------------------------------- #
# Reading and describing the data
# --------------------------------------------------------------------------- #

@dataclass
class Recording:
    """A TRACES file and the timing facts derived from it."""

    filename: str
    data: pd.DataFrame

    @property
    def n_points(self) -> int:
        return len(self.data.index)

    @property
    def first_hour(self) -> float:
        return float(self.data["Hours"].iloc[0])

    @property
    def last_hour(self) -> float:
        return float(self.data["Hours"].iloc[-1])

    @property
    def duration_hours(self) -> float:
        return self.last_hour - self.first_hour

    @property
    def time_interval(self) -> float:
        """Average spacing between samples, in hours."""
        return self.duration_hours / (self.n_points - 1)

    def describe(self) -> None:
        print(f"Number of rows: {self.n_points}")
        print(f"First time point: {self.first_hour} h")
        print(f"Last time point: {self.last_hour} h")
        print(
            f"Total time duration: {self.duration_hours} h "
            f"= {self.duration_hours / HOURS_PER_DAY} days"
        )
        print(f"Average time interval: {self.time_interval} h")


def read_traces(path: str) -> Recording:
    """Read a tab-separated TRACES.nnn file (3 header lines, 1 footer line)."""
    data = pd.read_csv(
        path,
        header=None,
        sep="\t",
        skiprows=3,
        skipfooter=1,
        index_col=False,
        names=COLUMN_NAMES,
        engine="python",
    )
    return Recording(filename=path, data=data.loc[:, COLUMN_NAMES])


def rolling_mean(data: pd.DataFrame, window: int) -> pd.DataFrame:
    """Centered moving average of the channel columns; 'Hours' is left alone.

    (Smoothing 'Hours' as well would shift the time axis at the edges, where
    min_periods=1 averages over an incomplete window.)
    """
    smoothed = data[CHANNELS].rolling(window=window, center=True, min_periods=1).mean()
    smoothed.insert(0, "Hours", data["Hours"])
    return smoothed


# --------------------------------------------------------------------------- #
# Detrending
# --------------------------------------------------------------------------- #

@dataclass
class Trend:
    """A trend line plus a human-readable description of how it was made."""

    data: pd.DataFrame
    label: str          # shown in plot legends
    short_label: str    # used in the Excel sheet name (31-character limit)
    summary: str        # shown on the 'Note' sheet


def moving_average_trend(
    data: pd.DataFrame, window_hours: float, time_interval: float
) -> Trend:
    n_points = math.ceil(window_hours / time_interval)
    if n_points % 2 == 0:
        n_points += 1
    window_span = time_interval * (n_points - 1)
    print(
        f"Window size for trend line (Moving Average): "
        f"{window_span:.6f} h ({n_points} points)"
    )

    trend = rolling_mean(data, n_points).bfill().ffill()
    return Trend(
        data=trend,
        label=f"{n_points}-point moving average, centered",
        short_label=f"{n_points}PMA",
        summary=f"{window_hours}h window ({n_points} points)",
    )


def sinc_filter_trend(
    data: pd.DataFrame, cutoff_period_hours: float, order: int, time_interval: float
) -> Trend:
    n_points = len(data.index)
    sampling_rate = 1.0 / time_interval                 # samples per hour
    cutoff_frequency = 1.0 / cutoff_period_hours        # cycles per hour
    nyquist_frequency = 0.5 * sampling_rate
    norm_cutoff = cutoff_frequency / nyquist_frequency

    # filtfilt pads the signal by 3 * len(taps), so the order has to stay well
    # below the record length; it also has to be odd for a zero-phase lowpass.
    max_order = int(n_points / 3) - 1
    if order >= max_order:
        print(
            f"Warning: Sinc Filter Order ({order}) is too high for data length "
            f"({n_points} points)."
        )
        order = max(1, max_order)
        print(f"Adjusting Sinc Filter Order to {order} for stability with filtfilt.")
    if order % 2 == 0:
        order += 1

    print(f"Sampling Rate: {sampling_rate:.2f} samples/hour")
    print(f"Cutoff Period for Sinc Filter: {cutoff_period_hours} hours")
    print(f"Cutoff Frequency: {cutoff_frequency:.4f} cycles/hour")
    print(f"Normalized Cutoff Frequency: {norm_cutoff:.4f}")
    print(f"Sinc Filter Order: {order}")

    taps = signal.firwin(order, norm_cutoff, pass_zero="lowpass")

    trend = data.copy()
    for channel in CHANNELS:
        if not data[channel].notna().any():
            trend[channel] = np.nan
            continue

        # filtfilt cannot handle NaNs; interpolate across gaps first.
        filled = data[channel].interpolate(method="linear", limit_direction="both")
        try:
            trend[channel] = signal.filtfilt(taps, [1.0], filled)
        except ValueError as error:
            print(f"Warning: could not filter channel {channel}: {error}")
            print("Consider reducing the filter order or using a longer recording.")
            trend[channel] = np.nan

    return Trend(
        data=trend,
        label=f"Sinc Filter C={cutoff_period_hours}h O={order}",
        short_label=f"Sinc C={cutoff_period_hours}h O={order}",
        summary=f"{cutoff_period_hours}h cutoff, order {order}",
    )


def build_trend(
    data: pd.DataFrame,
    method: str,
    time_interval: float,
    *,
    window_hours: float,
    cutoff_period_hours: float,
    order: int,
) -> Trend:
    if method == "Moving Average":
        return moving_average_trend(data, window_hours, time_interval)
    if method == "Sinc Filter":
        return sinc_filter_trend(data, cutoff_period_hours, order, time_interval)
    raise ValueError(f"Unknown detrending method: {method!r}")


def detrend(data: pd.DataFrame, trend: pd.DataFrame) -> pd.DataFrame:
    """Subtract the trend from the channels, keeping the original time axis."""
    detrended = data[CHANNELS] - trend[CHANNELS]
    detrended.insert(0, "Hours", data["Hours"])
    return detrended


# --------------------------------------------------------------------------- #
# Peaks and troughs
# --------------------------------------------------------------------------- #

def find_peaks_and_troughs(
    series: pd.Series,
    min_separation_hours: float = PEAK_MIN_SEPARATION_HOURS,
    time_interval: float | None = None,
):
    """Return the series indices of the peaks and of the troughs."""
    valid = series.dropna()
    if valid.empty:
        empty = pd.Index([])
        return empty, empty

    if min_separation_hours and time_interval:
        distance = max(1, int(min_separation_hours / time_interval))
    else:
        distance = 5

    peaks, _ = signal.find_peaks(valid.to_numpy(), distance=distance)
    troughs, _ = signal.find_peaks(-valid.to_numpy(), distance=distance)
    return valid.index[peaks], valid.index[troughs]


def peaks_and_troughs_table(smoothed: pd.DataFrame, time_interval: float) -> pd.DataFrame:
    """One row per detected peak/trough across all channels."""
    rows = []
    for channel in CHANNELS:
        peaks, troughs = find_peaks_and_troughs(
            smoothed[channel], PEAK_MIN_SEPARATION_HOURS, time_interval
        )
        for kind, indices in (("Peak", peaks), ("Trough", troughs)):
            for index in indices:
                rows.append(
                    {
                        "Channel": channel,
                        "Type": kind,
                        "Time (Hours)": smoothed.loc[index, "Hours"],
                    }
                )
    return pd.DataFrame(rows, columns=["Channel", "Type", "Time (Hours)"])


# --------------------------------------------------------------------------- #
# Excel / .dat / zip output
# --------------------------------------------------------------------------- #

def build_note_sheet(rows: list[tuple[str, object]]) -> pd.DataFrame:
    """Labels in column A, values in column E (as in the original workbook)."""
    blank = [""] * len(rows)
    return pd.DataFrame(
        {
            "A": [label for label, _ in rows],
            "B": blank,
            "C": blank,
            "D": blank,
            "E": [value for _, value in rows],
        }
    )


def write_main_workbook(
    path: str,
    note: pd.DataFrame,
    raw: pd.DataFrame,
    smoothed: dict[int, pd.DataFrame],
    trend: Trend,
    detrended: pd.DataFrame,
    detrended_smoothed: dict[int, pd.DataFrame],
    peaks_troughs: pd.DataFrame,
) -> None:
    with pd.ExcelWriter(path) as writer:
        note.to_excel(writer, sheet_name="Note", index=False, header=False)
        raw.to_excel(writer, sheet_name="Raw Data")
        for window, frame in smoothed.items():
            frame.to_excel(
                writer, sheet_name=sheet_name(f"{window}-point moving average ({window}PMA)")
            )
        trend.data.to_excel(writer, sheet_name=sheet_name(f"Trend line ({trend.short_label})"))
        detrended.to_excel(writer, sheet_name="Detrended Data")
        for window, frame in detrended_smoothed.items():
            frame.to_excel(writer, sheet_name=sheet_name(f"Detrended Data {window}PMA"))

        for channel in CHANNELS:
            per_channel = pd.DataFrame(
                {
                    "Hours": raw["Hours"],
                    "Raw_data": raw[channel],
                    **{f"{w}PMA": smoothed[w][channel] for w in smoothed},
                    "trend_line": trend.data[channel],
                    "detrended_data": detrended[channel],
                    **{
                        f"detrended_data_{w}PMA": detrended_smoothed[w][channel]
                        for w in detrended_smoothed
                    },
                }
            )
            per_channel.to_excel(writer, sheet_name=f"Channel {channel}")

        if peaks_troughs.empty:
            print("No peaks or troughs detected for any channel.")
        else:
            peaks_troughs.to_excel(writer, sheet_name="Peaks and Troughs", index=False)


def write_raw_only_workbook(path: str, raw: pd.DataFrame, time_interval: float) -> None:
    """Single-sheet workbook for pyBOAT; the interval is encoded in the sheet name."""
    name = sheet_name(f"INTVL = {time_interval:.15f} h")
    with pd.ExcelWriter(path) as writer:
        raw.to_excel(writer, sheet_name=name, index=False, header=True)


def write_dat_files(raw: pd.DataFrame, directory: Path) -> list[Path]:
    """One LumiCycle-readable .dat file per channel: days<TAB>value."""
    days = raw["Hours"] / HOURS_PER_DAY
    written = []
    for channel in CHANNELS:
        path = directory / f"{channel}.dat"
        pd.DataFrame({"Days": days, channel: raw[channel]}).to_csv(
            path, header=False, index=False, sep="\t"
        )
        written.append(path)
    return written


def zip_files(paths: list[Path], archive_path: Path) -> None:
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in paths:
            archive.write(path, arcname=path.name)


def remove_previous_outputs(directory: Path, keep: set[str] | None = None) -> None:
    """Delete output files left over from an earlier run of this notebook."""
    keep = keep or set()
    patterns = ("*.xlsx", "*.pdf", "*.dat", "*.zip", "TRACES.*", "Traces.*", "traces.*")
    for pattern in patterns:
        for path in directory.glob(pattern):
            if path.name not in keep:
                path.unlink(missing_ok=True)


# --------------------------------------------------------------------------- #
# Plotting
# --------------------------------------------------------------------------- #

OVERVIEW_RC = {
    "font.size": 5,
    "axes.titlesize": 4,
    "axes.labelsize": 4,
    "xtick.labelsize": 4,
    "ytick.labelsize": 4,
    "legend.fontsize": 3,
    "figure.titlesize": 5,
}

CHANNEL_PAGE_RC = {
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 6,
    "figure.titlesize": 12,
}

FIT_PAGE_RC = {
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6,
}


@dataclass
class AxisStyle:
    """Shared x-axis settings for every time-series plot."""

    x_min: int
    x_max: int
    ticks: list[int]
    minor_tick: float
    show_minor: bool

    def apply(self, ax, x_min: float | None = None) -> None:
        ax.set_xlim(self.x_min if x_min is None else x_min, self.x_max)
        ax.set_xticks(self.ticks)
        if self.show_minor:
            ax.xaxis.set_minor_locator(MultipleLocator(self.minor_tick))
        ax.grid(True, linewidth=0.5, color="lightgray", linestyle="--")


def make_axis_style(hours: pd.Series, major_tick: int, minor_tick: float) -> AxisStyle:
    x_min = int(math.floor(hours.min() / HOURS_PER_DAY)) * 24
    x_max = int(math.ceil(hours.max() / 12)) * 12 + 12
    return AxisStyle(
        x_min=x_min,
        x_max=x_max,
        ticks=list(range(x_min, x_max, major_tick)),
        minor_tick=minor_tick,
        show_minor=bool(minor_tick) and minor_tick < major_tick,
    )


def plot_overview_grid(
    pdf: PdfPages,
    *,
    title: str,
    channel_order: list[str],
    scatter: pd.DataFrame,
    line: pd.DataFrame,
    line_label: str,
    label_suffix: str,
    y_axis_label: str,
    axis: AxisStyle,
    rows: int,
    cols: int,
    start_y_at_zero: bool,
) -> None:
    """One page with a small panel per channel."""
    with plt.rc_context(OVERVIEW_RC):
        figure = plt.figure(figsize=(11, 8.5))
        figure.subplots_adjust(hspace=0.15)
        figure.suptitle(title, fontsize=12)

        for position, channel in enumerate(channel_order, start=1):
            ax = figure.add_subplot(rows, cols, position)
            label = f"Ch # {channel}{label_suffix}"
            ax.scatter(scatter["Hours"], scatter[channel], s=0.1, c="blue", label=label)
            ax.plot(
                line["Hours"], line[channel], "-r", linewidth=0.5, label=line_label
            )
            axis.apply(ax, x_min=axis.x_min - 6)
            if start_y_at_zero:
                ax.set_ylim(0, upper_ylimit(scatter[channel]))
            ax.legend(loc="upper right", fontsize=4)

        figure.text(0.50, 0.06, "Time (hours)", ha="center", fontsize=10)
        figure.text(
            0.08, 0.50, y_axis_label, ha="center", va="center",
            rotation="vertical", fontsize=10,
        )
        pdf.savefig(figure)
        plt.show()
        plt.close(figure)


def _draw_actogram(
    ax,
    smoothed: pd.DataFrame,
    peaks,
    troughs,
    day_length: float,
    last_hour: float,
    show_labels: bool,
) -> None:
    """Double-plotted actogram of peak and trough times."""

    def to_actogram_points(indices):
        points = []
        for hour in smoothed.loc[indices, "Hours"].to_numpy():
            day = hour // day_length
            time_in_day = hour - day * day_length
            points.append((time_in_day, day, hour))
            if day > 0:  # second copy of the day, shifted one cycle right
                points.append((time_in_day + day_length, day - 1, hour))
        return points

    for indices, color, label in (
        (peaks, "red", "Peaks"),
        (troughs, "blue", "Troughs"),
    ):
        points = to_actogram_points(indices)
        if not points:
            continue
        xs, ys, original_hours = zip(*points)
        ax.scatter(xs, ys, marker="o", s=20, color=color, label=label)
        if show_labels:
            for x, y, hour in zip(xs, ys, original_hours):
                ax.text(x + 0.5, y, f"{hour:.2f}", fontsize=5, color=color)

    if show_labels:  # leave room for the text labels
        ax.set_xlim(-0.085 * day_length, 2.085 * day_length)
    else:
        ax.set_xlim(0, 2 * day_length)

    ax.set_xticks([fraction * day_length for fraction in np.arange(0, 2.25, 0.25)])

    # Enough decimal places that the tick labels stay distinguishable.
    scaled = day_length if float(day_length).is_integer() else day_length * 10
    extra = 0 if float(day_length).is_integer() else 1
    if scaled % 4 == 0:
        decimals = 0 + extra
    elif scaled % 2 == 0:
        decimals = 1 + extra
    else:
        decimals = 2 + extra
    ax.xaxis.set_major_formatter(FormatStrFormatter(f"%.{decimals}f"))

    if day_length == HOURS_PER_DAY:
        ax.set_title("Peaks and Troughs", fontsize=10)
        ax.set_ylabel("Days", fontsize=10)
    else:
        ax.set_title(
            f"Peaks and Troughs (scaled x-axis: T = {day_length} hours, T{day_length})",
            fontsize=10,
        )
        ax.set_ylabel(f"Days (T{day_length})", fontsize=10)

    n_days = max(last_hour // day_length, 1)
    ax.set_ylim(-n_days * 0.05, n_days * 1.05)
    ax.yaxis.set_major_locator(MultipleLocator(base=1))
    ax.set_xlabel("Time (Hours)", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.7)
    if ax.get_legend_handles_labels()[0]:  # nothing detected on a flat channel
        ax.legend(loc="upper right", fontsize=6)
    ax.invert_yaxis()  # day 0 at the top


def plot_channel_page(
    pdf: PdfPages,
    *,
    channel: str,
    experiment_number: str,
    raw: pd.DataFrame,
    trend: Trend,
    detrended: pd.DataFrame,
    detrended_smoothed: pd.DataFrame,
    axis: AxisStyle,
    time_interval: float,
    day_length: float,
    last_hour: float,
    label_peaks: bool,
    label_actogram: bool,
) -> None:
    """One page per channel: raw + trend, detrended + peaks, actogram."""
    with plt.rc_context(CHANNEL_PAGE_RC):
        figure = plt.figure(figsize=(8.5, 11))
        figure.suptitle(f"{experiment_number}   Ch # {channel}", fontsize=12)

        # --- raw data with its trend line ---------------------------------- #
        ax_raw = figure.add_subplot(3, 1, 1)
        ax_raw.scatter(
            raw["Hours"], raw[channel], s=3.0, c="violet", label="Bioluminescence"
        )
        ax_raw.plot(
            trend.data["Hours"], trend.data[channel], "-r", linewidth=1.0,
            label=f"Trend line ({trend.label})",
        )
        axis.apply(ax_raw)
        ax_raw.set_ylim(0, upper_ylimit(raw[channel]))
        ax_raw.set_xlabel("Hours", fontsize=10)
        ax_raw.set_ylabel("Bioluminescence", fontsize=10)
        ax_raw.legend(loc="upper right", fontsize=5)

        # --- detrended data with peaks and troughs -------------------------- #
        ax_detrended = figure.add_subplot(3, 1, 2)
        ax_detrended.scatter(
            detrended["Hours"], detrended[channel], s=3.0, c="violet",
            label="Detrended bioluminescence",
        )
        ax_detrended.plot(
            detrended_smoothed["Hours"], detrended_smoothed[channel], "-b",
            linewidth=1.0, label="Smoothed line (9-point moving average, centered)",
        )

        peaks, troughs = find_peaks_and_troughs(
            detrended_smoothed[channel], PEAK_MIN_SEPARATION_HOURS, time_interval
        )
        y_span = None
        for indices, color, label, offset_sign, va in (
            (peaks, "red", "Peaks", 1, "bottom"),
            (troughs, "blue", "Troughs", -1, "top"),
        ):
            if len(indices) == 0:
                continue
            hours = detrended_smoothed.loc[indices, "Hours"]
            values = detrended_smoothed.loc[indices, channel]
            ax_detrended.scatter(hours, values, marker="o", s=30, color=color, label=label)
            if not label_peaks:
                continue
            if y_span is None:  # measured once, after both scatters are drawn
                bottom, top = ax_detrended.get_ylim()
                y_span = top - bottom
            for hour, value in zip(hours, values):
                ax_detrended.text(
                    hour, value + offset_sign * y_span * 0.05, f"{hour:.2f} h",
                    fontsize=5, color=color, ha="center", va=va,
                )

        axis.apply(ax_detrended)
        ax_detrended.set_xlabel("Hours", fontsize=10)
        ax_detrended.set_ylabel("Detrended bioluminescence", fontsize=10)
        ax_detrended.legend(loc="upper right", fontsize=5)

        # --- actogram ------------------------------------------------------- #
        ax_actogram = figure.add_subplot(3, 1, 3)
        _draw_actogram(
            ax_actogram, detrended_smoothed, peaks, troughs,
            day_length, last_hour, label_actogram,
        )

        figure.tight_layout(rect=(0, 0.02, 1, 0.98))
        pdf.savefig(figure)
        plt.show()
        plt.close(figure)


# --------------------------------------------------------------------------- #
# Damped sine fitting
# --------------------------------------------------------------------------- #

FIT_COLUMNS = [
    "Channel",
    "Fitted Amplitude",
    "Fitted Period (Hours)",
    "Fitted Phase (radians)",
    "Fitted Decay Rate",
    "Fitted Offset",
]


def damped_sine(t, amplitude, period, phase, decay_rate, offset):
    return amplitude * np.exp(-decay_rate * t) * np.sin(2 * np.pi * t / period + phase) + offset


def _empty_fit(channel: str) -> dict:
    return {"Channel": channel, **{column: np.nan for column in FIT_COLUMNS[1:]}}


def fit_damped_sine(times: pd.Series, values: pd.Series) -> tuple[np.ndarray, float]:
    """Fit the damped sine model. Returns the parameters and the normalized phase."""
    amplitude_guess = (values.max() - values.min()) / 2.0
    if amplitude_guess <= 0:
        amplitude_guess = values.std() * 2
    if not amplitude_guess:
        amplitude_guess = 1.0

    guesses = [amplitude_guess, HOURS_PER_DAY, 0.0, 0.01, values.mean()]
    lower = [0, FIT_MIN_PERIOD_HOURS, -2 * np.pi, 0, -np.inf]
    upper = [np.inf, FIT_MAX_PERIOD_HOURS, 2 * np.pi, 0.5, np.inf]

    params, _ = curve_fit(
        damped_sine, times, values, p0=guesses, bounds=(lower, upper), maxfev=10000
    )
    phase = float(params[2]) % (2 * np.pi)
    return params, phase


def plot_fit_page(
    pdf: PdfPages,
    *,
    channel: str,
    experiment_number: str,
    raw: pd.DataFrame,
    trend: Trend,
    detrended: pd.DataFrame,
    fit_times: pd.Series,
    fit_values: pd.Series,
    model_times: pd.Series,
    params: np.ndarray,
    phase: float,
    axis: AxisStyle,
    start_hour: float,
    end_hour: float,
) -> None:
    amplitude, period, _, decay_rate, offset = params
    fitted = damped_sine(model_times, *params)

    with plt.rc_context(FIT_PAGE_RC):
        figure, (ax_raw, ax_detrended) = plt.subplots(
            2, 1, figsize=(8.5, 11), sharex=True
        )
        figure.suptitle(
            f"{experiment_number} - Damped Sine Fit Channel {channel}", fontsize=12
        )

        # The raw-scale fit is the detrended fit put back on top of the trend.
        raw_fit = fitted + trend.data.loc[fit_times.index, channel]
        ax_raw.plot(
            raw["Hours"], raw[channel], "o", markersize=2, color="gray",
            label="Full Raw Data",
        )
        ax_raw.plot(
            trend.data["Hours"], trend.data[channel], "k--", linewidth=1.0,
            label="Trend Line",
        )
        ax_raw.plot(fit_times, raw_fit, "r-", linewidth=1.5, label="Raw Data Fit")
        ax_raw.set_ylabel("Bioluminescence", fontsize=10)
        ax_raw.set_title(
            f"Raw Data Fit (P={period:.2f}h A={amplitude:.2f} Ph={phase:.2f})",
            fontsize=11,
        )
        ax_raw.set_ylim(bottom=0)
        ax_raw.legend(loc="upper right", fontsize=8)
        ax_raw.grid(True, linestyle="--", alpha=0.7)

        ax_detrended.plot(
            detrended["Hours"], detrended[channel], "o", markersize=2, color="gray",
            label="Full Detrended Data",
        )
        ax_detrended.plot(
            fit_times, fit_values, "o", markersize=2, color="blue",
            label=f"Data for Fitting (from {start_hour}h to {end_hour}h)",
        )
        ax_detrended.plot(
            fit_times, fitted, "r-", linewidth=1.5, label="Damped Sine Fit"
        )
        ax_detrended.set_xlabel("Time (Hours)", fontsize=10)
        ax_detrended.set_ylabel("Detrended Bioluminescence", fontsize=10)
        ax_detrended.set_title(
            f"Detrended Data Fit (Dec={decay_rate:.6f} Off={offset:.6f})", fontsize=11
        )
        ax_detrended.legend(loc="upper right", fontsize=8)
        ax_detrended.grid(True, linestyle="--", alpha=0.7)
        ax_detrended.set_xlim(axis.x_min, axis.x_max)
        ax_detrended.xaxis.set_major_locator(MultipleLocator(base=24))
        ax_detrended.xaxis.set_minor_locator(MultipleLocator(base=12))

        figure.tight_layout(rect=(0, 0.03, 1, 0.95))
        pdf.savefig(figure)
        plt.show()
        plt.close(figure)


def fit_all_channels(
    pdf: PdfPages,
    *,
    channel_order: list[str],
    experiment_number: str,
    raw: pd.DataFrame,
    trend: Trend,
    detrended: pd.DataFrame,
    axis: AxisStyle,
    start_hour: float,
    end_hour: float,
) -> pd.DataFrame:
    window = detrended[
        detrended["Hours"].between(start_hour, end_hour)
    ]

    results = []
    for channel in channel_order:
        values = window[channel].dropna()
        times = window.loc[values.index, "Hours"]

        if len(values) < FIT_MIN_POINTS:
            print(
                f"Skipping damped sine fit for Channel {channel}: only {len(values)} "
                f"points in the fitting window (need {FIT_MIN_POINTS})."
            )
            results.append(_empty_fit(channel))
            continue

        # The model's decay term assumes t starts at 0.
        model_times = times - times.min()
        try:
            params, phase = fit_damped_sine(model_times, values)
        except (RuntimeError, ValueError) as error:
            print(f"Could not fit damped sine to Channel {channel}: {error}")
            results.append(_empty_fit(channel))
            continue

        amplitude, period, _, decay_rate, offset = params
        results.append(
            {
                "Channel": channel,
                "Fitted Amplitude": amplitude,
                "Fitted Period (Hours)": period,
                "Fitted Phase (radians)": phase,
                "Fitted Decay Rate": decay_rate,
                "Fitted Offset": offset,
            }
        )

        plot_fit_page(
            pdf,
            channel=channel,
            experiment_number=experiment_number,
            raw=raw,
            trend=trend,
            detrended=detrended,
            fit_times=times,
            fit_values=values,
            model_times=model_times,
            params=params,
            phase=phase,
            axis=axis,
            start_hour=start_hour,
            end_hour=end_hour,
        )

    return pd.DataFrame(results, columns=FIT_COLUMNS)


print("Helper functions loaded." + ("" if IN_COLAB else "  (not running on Colab)"))


###############################################################################
# CELL 3 - Run: upload the TRACES file and generate the output files
###############################################################################

# @title Run: upload the TRACES file and generate the output files { display-mode: "form" }

# --------------------------------------------------------------------------- #
# Turn the settings above into the values the code below uses
# --------------------------------------------------------------------------- #

WORKING_DIRECTORY = Path.cwd()

MAJOR_TICK_HOURS = {"Every 12 hours": 12, "Every 24 hours": 24, "Every 48 hours": 48}
MINOR_TICK_HOURS = {
    "No minor ticks": 0,
    "Every 2 hours": 2,
    "Every 4 hours": 4,
    "Every 6 hours": 6,
    "Every 12 hours": 12,
    "Every 24 hours": 24,
}
SUBPLOT_GRID = {"3-column by 10-row": (10, 3), "4-column by 8-row": (8, 4)}

major_tick = MAJOR_TICK_HOURS[Major_Ticks]
minor_tick = MINOR_TICK_HOURS[Minor_Ticks]
grid_rows, grid_cols = SUBPLOT_GRID[Subplot_Matrix]

# Channel 00 is the reference channel, so it is plotted either first or last.
channel_order = CHANNELS if Data_Plotting.endswith("first") else CHANNELS[1:] + CHANNELS[:1]

label_peaks = Label_peaks_and_troughs_in_detrended_data_plot == "Yes"
label_actogram = Label_peaks_and_troughs_in_actogram == "Yes"
day_length = float(Actogram_X_axis_scale)

start_hour, end_hour = Start_Hour_for_Fitting, End_Hour_for_Fitting
if end_hour - start_hour <= 23:  # too short to see a full cycle; fall back
    start_hour, end_hour = 24, 120
    print(f"Fitting window was shorter than 24 h; using {start_hour}-{end_hour} h instead.")

# --------------------------------------------------------------------------- #
# Read the data
# --------------------------------------------------------------------------- #

start_time = utc_now()

if IN_COLAB:
    remove_previous_outputs(WORKING_DIRECTORY)
    uploaded = colab_files.upload()
    traces_filename = next(iter(uploaded))
else:  # local kernel: pick up a TRACES file sitting next to the notebook
    candidates = sorted(WORKING_DIRECTORY.glob("[Tt][Rr][Aa][Cc][Ee][Ss].*"))
    if not candidates:
        raise FileNotFoundError("No TRACES.nnn file found in the working directory.")
    traces_filename = candidates[0].name
    remove_previous_outputs(WORKING_DIRECTORY, keep={traces_filename})

print(f"\nStarted at {start_time:%Y-%m-%d %H:%M:%S} (UTC)")
print("\nReading the data...")

recording = read_traces(traces_filename)
raw = recording.data
time_interval = recording.time_interval

print("\nRaw Data:")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(raw)
recording.describe()

# --------------------------------------------------------------------------- #
# Smooth and detrend
# --------------------------------------------------------------------------- #

print(f"\nCalculating trend line and detrended data using {Detrending_Method}...")

smoothed = {window: rolling_mean(raw, window) for window in SMOOTHING_WINDOWS}

trend = build_trend(
    raw,
    Detrending_Method,
    time_interval,
    window_hours=Window_size_for_trend_line_moving_average,
    cutoff_period_hours=Sinc_Filter_Cutoff_Period_Hours,
    order=Sinc_Filter_Order,
)

detrended = detrend(raw, trend.data)
detrended_smoothed = {window: rolling_mean(detrended, window) for window in SMOOTHING_WINDOWS}
detrended_9pma = detrended_smoothed[9]

# --------------------------------------------------------------------------- #
# Excel, .dat and zip output
# --------------------------------------------------------------------------- #

print("\nGenerating an Excel file...")

note = build_note_sheet(
    [
        ("Experiment Number", Experiment_number),
        ("Experiment Title", Experiment_title),
        ("Experiment Start Date", Date_experiment_started),
        ("TRACES File", traces_filename),
        ("", ""),
        ("Number of Time Points", recording.n_points),
        ("Total Time Duration (Hours)", recording.duration_hours),
        ("Average Time Interval (Hours)", time_interval),
        ("Detrending Method", Detrending_Method),
        ("Trend Line Parameter", trend.summary),
        ("", ""),
        ("Data Processed Date and Time (UTC)", utc_now_string()),
    ]
)

main_workbook = f"{Experiment_number}_data.xlsx"
write_main_workbook(
    main_workbook,
    note=note,
    raw=raw,
    smoothed=smoothed,
    trend=trend,
    detrended=detrended,
    detrended_smoothed=detrended_smoothed,
    peaks_troughs=peaks_and_troughs_table(detrended_9pma, time_interval),
)
print(f"\nExcel file: {main_workbook}  has been generated.")

raw_workbook = f"{Experiment_number}_data_2.xlsx"
write_raw_only_workbook(raw_workbook, raw, time_interval)
print(f"\nExcel file: {raw_workbook}  has been generated.")

print("\nGenerating a data file for each channel (.dat files)...")
dat_files = write_dat_files(raw, WORKING_DIRECTORY)

print("\nPacking .dat files into a zip file...")
zip_archive = WORKING_DIRECTORY / f"{Experiment_number}_data.zip"
zip_files(dat_files, zip_archive)

# --------------------------------------------------------------------------- #
# Plots
# --------------------------------------------------------------------------- #

print("\nPlotting...")
plot_pdf = f"{Experiment_number}_data_plots.pdf"
axis = make_axis_style(raw["Hours"], major_tick, minor_tick)
plt.rcParams.update({"figure.max_open_warning": 0})

with PdfPages(plot_pdf) as pdf:
    plot_overview_grid(
        pdf,
        title=Experiment_number,
        channel_order=channel_order,
        scatter=raw,
        line=trend.data,
        line_label=f"trend line ({Detrending_Method})",
        label_suffix="",
        y_axis_label="Bioluminescence",
        axis=axis,
        rows=grid_rows,
        cols=grid_cols,
        start_y_at_zero=True,
    )

    plot_overview_grid(
        pdf,
        title=f"{Experiment_number} - detrended data ({Detrending_Method})",
        channel_order=channel_order,
        scatter=detrended,
        line=detrended_smoothed[5],
        line_label="5-point moving average",
        label_suffix=" - detrend",
        y_axis_label="Detrended Bioluminescence",
        axis=axis,
        rows=grid_rows,
        cols=grid_cols,
        start_y_at_zero=False,
    )

    print("\nPlotting individual channels with actograms...")
    for channel in channel_order:
        plot_channel_page(
            pdf,
            channel=channel,
            experiment_number=Experiment_number,
            raw=raw,
            trend=trend,
            detrended=detrended,
            detrended_smoothed=detrended_9pma,
            axis=axis,
            time_interval=time_interval,
            day_length=day_length,
            last_hour=recording.last_hour,
            label_peaks=label_peaks,
            label_actogram=label_actogram,
        )

    print("\nPerforming damped sine curve fitting...")
    fit_results = fit_all_channels(
        pdf,
        channel_order=channel_order,
        experiment_number=Experiment_number,
        raw=raw,
        trend=trend,
        detrended=detrended,
        axis=axis,
        start_hour=start_hour,
        end_hour=end_hour,
    )

print("\nDamped Sine Curve Fitting Results:")
display(fit_results)

fit_workbook = f"{Experiment_number}_damped_sine_fit_results.xlsx"
with pd.ExcelWriter(fit_workbook) as writer:
    fit_results.to_excel(writer, sheet_name="Damped Sine Fit", index=False)
print(f"\nDamped sine fitting results written to: {fit_workbook}")

# --------------------------------------------------------------------------- #
# Download
# --------------------------------------------------------------------------- #

output_files = [main_workbook, raw_workbook, plot_pdf, zip_archive.name, fit_workbook]
if IN_COLAB:
    for name in output_files:
        colab_files.download(name)
else:
    print("\nOutput files written to " + str(WORKING_DIRECTORY) + ":")
    for name in output_files:
        print("  " + name)

end_time = utc_now()
print(f"\nElapsed time: {(end_time - start_time).seconds} seconds")
print(f"Completed at {end_time:%Y-%m-%d %H:%M:%S} (UTC)\n")
